# Heckman's Sample-Selection Model (Heckit)

When the outcome of interest is observed only for individuals who 'self-select' into the sample (think wages — observed only for those who work) and that selection is correlated with the outcome error, OLS on the selected subsample is biased. Heckman (1979) introduced a two-step correction; the joint maximum-likelihood estimator is its asymptotically efficient sibling.

Model:

$$Y^* = X'\beta + e, \quad S^* = Z'\gamma + u, \quad S = \mathbf 1\{S^* > 0\},$$
$$Y = Y^* \text{ if } S = 1, \text{ missing otherwise}, \quad (e, u) \sim \mathcal N\!\left(0, \begin{pmatrix}\sigma^2 & \rho\sigma \\ \rho\sigma & 1\end{pmatrix}\right).$$

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from censtrunc import HeckitRegression

rng = np.random.default_rng(42)
n = 4000
# Shared regressor (in both equations); two exclusive regressors
shared  = rng.normal(size=n)
x_only  = rng.normal(size=n)       # in the outcome eq. only
z_only  = rng.normal(size=n)       # in the selection eq. only (exclusion)

beta_true  = np.array([1.0,  0.5, -0.3])     # const, shared, x_only
gamma_true = np.array([0.0,  0.3,  0.6])     # const, shared, z_only
rho_true, sigma_true = 0.6, 1.0

Sigma = np.array([[sigma_true**2, rho_true*sigma_true],
                  [rho_true*sigma_true, 1.0]])
errs = rng.multivariate_normal([0.0, 0.0], Sigma, size=n)
e, u = errs[:, 0], errs[:, 1]

X = np.column_stack([shared, x_only])
Z = np.column_stack([shared, z_only])
S = (gamma_true[0] + Z @ gamma_true[1:] + u > 0).astype(int)
y_full = beta_true[0] + X @ beta_true[1:] + e
y = np.where(S == 1, y_full, np.nan)
print(f'Selected (S=1): {S.sum()} / {n}  ({S.mean():.1%})')

Selected (S=1): 2022 / 4000  (50.5%)


## Why OLS on the selected sample is biased

Because $u$ and $e$ are correlated ($\rho > 0$), individuals with high $e$ tend to have high $u$ and therefore are more likely to be selected. The conditional mean for the selected sample is

$$\mathbb E[Y \mid X, S = 1] = X'\beta + \rho\sigma\,\lambda(Z'\gamma),$$

with $\lambda(\cdot)$ the inverse Mills ratio. Omitting $\lambda(Z'\gamma)$ from the regression — what naive OLS does — produces omitted-variable bias on $\beta$.

In [2]:
sel = ~np.isnan(y)
X_full = sm.add_constant(X)
ols = sm.OLS(y[sel], X_full[sel]).fit()
print('OLS on selected subsample (biased):')
print(np.asarray(ols.params).round(4))
print('truth:', beta_true)

OLS on selected subsample (biased):
[ 1.3651  0.4366 -0.3037]
truth: [ 1.   0.5 -0.3]


## Two-step Heckit

Heckman's two-step recipe:

1. Probit of $S$ on $Z$ to obtain $\hat\gamma$.
2. Compute the inverse Mills ratio $\hat\lambda_i = \phi(Z_i'\hat\gamma)/\Phi(Z_i'\hat\gamma)$ for selected observations.
3. OLS of $y_i$ on $(X_i,\hat\lambda_i)$ for selected $i$. The coefficients are $\hat\beta$ and $\hat\rho\hat\sigma$.

In [3]:
m_two = HeckitRegression(method='twostep').fit(y, X, Z)
print(m_two.summary())

                         Heckman Selection Regression                         
Method:                  twostep                  No. Observations:        4000
Selected (S=1):          2022                     Censored (S=0):          1978
sigma_e:                 0.952857                 rho:                     0.531381
                     Outcome equation (Y* = X * beta + e)                     
------------------------------------------------------------------------------
                     coef     std err         z     P>|z|      [0.025    0.975]
------------------------------------------------------------------------------
const              1.0267      0.0491   20.9278    0.0000      0.9306    1.1229
x1                 0.4903      0.0223   21.9471    0.0000      0.4465    0.5341
x2                -0.3010      0.0209  -14.4109    0.0000     -0.3419   -0.2601
------------------------------------------------------------------------------
             Selection equation (S* = Z *

## Joint MLE Heckit

Maximise the full log-likelihood (Hansen 2022, §27.10):

$$\ell = \sum_{S_i=0} \log[1 - \Phi(Z_i'\gamma)] + \sum_{S_i=1}\!\left\{\log\Phi\!\left(\frac{Z_i'\gamma + (\rho/\sigma)(Y_i - X_i'\beta)}{\sqrt{1 - \rho^2}}\right) - \tfrac12\log(2\pi\sigma^2) - \frac{(Y_i - X_i'\beta)^2}{2\sigma^2}\right\}.$$

Asymptotically efficient, somewhat slower than the two-step.

In [4]:
m_ml = HeckitRegression(method='mle').fit(y, X, Z)
print(m_ml.summary())

                         Heckman Selection Regression                         
Method:                  mle                      No. Observations:        4000
Selected (S=1):          2022                     Censored (S=0):          1978
sigma_e:                 0.958607                 rho:                     0.552343
Log-Likelihood:          -4924.8374
                     Outcome equation (Y* = X * beta + e)                     
------------------------------------------------------------------------------
                     coef     std err         z     P>|z|      [0.025    0.975]
------------------------------------------------------------------------------
const              1.0108      0.0446   22.6544    0.0000      0.9234    1.0983
x1                 0.4925      0.0214   23.0340    0.0000      0.4506    0.5344
x2                -0.3017      0.0192  -15.7382    0.0000     -0.3393   -0.2642
------------------------------------------------------------------------------
     

## Side-by-side comparison

All three estimators on the same data. Heckit's two-step and MLE recover $\beta_{\text{shared}}$ much closer to the truth than OLS, and they recover $\rho$ and $\sigma$ as well.

In [5]:
comparison = pd.DataFrame({
    'true':         beta_true,
    'OLS (biased)': np.asarray(ols.params),
    'Heckit 2step': m_two.coef_,
    'Heckit MLE':   m_ml.coef_,
}, index=['const', 'shared', 'x_only'])
comparison['OLS error']   = comparison['OLS (biased)'] - comparison['true']
comparison['2step error'] = comparison['Heckit 2step']  - comparison['true']
comparison['MLE error']   = comparison['Heckit MLE']    - comparison['true']
comparison.round(4)

,true,OLS (biased),Heckit 2step,Heckit MLE,OLS error,2step error,MLE error
const,1.0,1.3651,1.0267,1.0108,0.3651,0.0267,0.0108
shared,0.5,0.4366,0.4903,0.4925,-0.0634,-0.0097,-0.0075
x_only,-0.3,-0.3037,-0.3010,-0.3017,-0.0037,-0.0010,-0.0017


In [6]:
pd.DataFrame({
    'true':      [rho_true, sigma_true],
    'two-step':  [m_two.rho_, m_two.sigma_],
    'MLE':       [m_ml.rho_,  m_ml.sigma_],
}, index=['rho', 'sigma']).round(4)

,true,two-step,MLE
rho,0.6,0.5314,0.5523
sigma,1.0,0.9529,0.9586


## Three kinds of prediction

- `kind='selection_prob'`  — $P(S=1 \mid Z) = \Phi(Z'\hat\gamma)$,
- `kind='outcome'`         — $\mathbb E[Y^* \mid X] = X'\hat\beta$ (unconditional),
- `kind='conditional'`     — $\mathbb E[Y \mid X, Z, S=1] = X'\hat\beta + \hat\rho\hat\sigma\,\lambda(Z'\hat\gamma)$.

In [7]:
preds = pd.DataFrame({
    'selection_prob': m_ml.predict(Z=Z[:6], kind='selection_prob'),
    'outcome':        m_ml.predict(X=X[:6], kind='outcome'),
    'conditional':    m_ml.predict(X=X[:6], Z=Z[:6], kind='conditional'),
    'observed_y':     y[:6],
    'selected':       S[:6],
})
preds.round(3)

,selection_prob,outcome,conditional,observed_y,selected
0,0.626,1.084,1.405,1.327,1
1,0.704,0.229,0.488,1.942,1
2,0.839,1.298,1.453,1.469,1
3,0.499,0.798,1.222,1.666,1
4,0.363,-0.381,0.166,-0.354,1
5,0.071,0.462,1.474,NaN,0


## Same fit via a patsy formula

Instead of building `y`, `X`, `Z` by hand, every model class accepts a `from_formula(...)` constructor in the statsmodels style. For Heckit you pass *two* formulas — outcome and selection — sharing a single dataframe:

In [8]:
df = pd.DataFrame({
    'shared': shared, 'x_only': x_only, 'z_only': z_only,
    'inlf':   S, 'lwage': y_full,
})
m_form = HeckitRegression.from_formula(
    outcome   = 'lwage ~ 1 + shared + x_only',
    selection = 'inlf  ~ 1 + shared + z_only',
    data=df, method='mle',
).fit()
print(pd.DataFrame({
    'beta (explicit)': m_ml.coef_,
    'beta (formula)':  m_form.coef_,
}, index=['const', 'shared', 'x_only']).round(6))

        beta (explicit)  beta (formula)
const          1.010834        1.010834
shared         0.492478        0.492478
x_only        -0.301747       -0.301747


## Bootstrap standard errors

Two-step second-stage standard errors are naive: they ignore the variability of $\hat\gamma$ from the probit step. A paired bootstrap gives proper SEs. The MLE Hessian-based SEs are already correct.

In [9]:
boot = m_two.bootstrap(y, X, Z, n_boot=80, seed=0)
pd.DataFrame({
    'beta':         m_two.coef_,
    'naive SE':     m_two.bse_,
    'bootstrap SE': boot['beta_se'],
}, index=['const', 'shared', 'x_only']).round(4)

,beta,naive SE,bootstrap SE
const,1.0267,0.0491,0.0479
shared,0.4903,0.0223,0.0214
x_only,-0.3010,0.0209,0.0205
